<a href="https://colab.research.google.com/github/gabiuxo/Algoritmos-de-Aprendizaje-Automatico/blob/main/practicaTema10parte1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice - Bias, Variance and Cross-Validation

This notebook completes the four challenges from Topic 10 (Part 1).

## Challenge 1 - Train vs. validation performance

A low training score suggests underfitting. A large gap between training and test scores suggests overfitting. Similar and high scores suggest a balanced model.

In [1]:
# classify simulated model results
results = [
    {'model': 'A', 'train': 0.99, 'test': 0.72},
    {'model': 'B', 'train': 0.65, 'test': 0.63},
    {'model': 'C', 'train': 0.92, 'test': 0.90}
]

for result in results:
    difference = result['train'] - result['test']

    if result['train'] < 0.70:
        diagnosis = 'underfitting (high bias)'
    elif difference > 0.10:
        diagnosis = 'overfitting (high variance)'
    else:
        diagnosis = 'balanced'

    print(f"model {result['model']}: {diagnosis}")

model A: overfitting (high variance)
model B: underfitting (high bias)
model C: balanced


## Challenge 2 - Decision tree complexity

The same dataset split is used for every depth. This makes the comparison fair.

In [2]:
# import the dataset and split it into train and test sets
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
x_train, x_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.20,
    random_state=42,
    stratify=data.target
)

for depth in [1, 4, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(x_train, y_train)

    train_score = model.score(x_train, y_train)
    test_score = model.score(x_test, y_test)

    print(f"depth={depth} -> train: {train_score:.3f} | test: {test_score:.3f}")

depth=1 -> train: 0.923 | test: 0.921
depth=4 -> train: 0.987 | test: 0.939
depth=None -> train: 1.000 | test: 0.912


**Interpretation:** depth 1 can underfit because it is very simple. Increasing depth can improve the model, but an unrestricted tree can memorize the training data and overfit.

## Challenge 3 - Stratified cross-validation

StratifiedKFold keeps a similar proportion of malignant and benign cases in every fold.

In [3]:
# evaluate logistic regression with stratified cross-validation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, data.data, data.target, cv=cv, scoring='accuracy')

print('accuracy by fold:', scores)
print(f'mean accuracy: {scores.mean():.4f}')

accuracy by fold: [0.97368421 0.94736842 0.96491228 0.99122807 0.99115044]
mean accuracy: 0.9737


## Challenge 4 - Mean and standard deviation

The mean reports average F1 performance. The standard deviation shows how much the F1 score changes from one fold to another; a lower value means more stable performance.

In [4]:
# calculate f1 scores and their dispersion
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    model,
    data.data,
    data.target,
    cv=cv,
    scoring='f1',
    return_train_score=False
)

test_scores = cv_results['test_score']
print('f1 by fold:', test_scores)
print(f'mean f1: {test_scores.mean():.4f} (+/- {test_scores.std():.4f})')

f1 by fold: [0.9787234  0.95945946 0.97297297 0.99300699 0.99300699]
mean f1: 0.9794 (+/- 0.0127)


## Final conclusion

Comparing training and test performance helps detect underfitting and overfitting. Cross-validation gives a more reliable estimate because it evaluates the model on several different partitions, while the standard deviation indicates whether that performance is stable.